# DhakaFlow Lab 01 — SUMO Foundation (Colab-only)
All SUMO installation and real execution happen in this Colab runtime. The notebook orchestrates reusable code from `src/`; it is not the implementation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Point to the active Lab 01 after Claude merges this handoff into the thesis repository
Preferred workflow: clone/pull the thesis repo into `/content/DhakaFlow-Nexus`, then keep code in Git and outputs in Drive.


In [ ]:
from pathlib import Path
import zipfile

# MODIFIED_BY_US (merge audit 2026-08-11): the thesis repo is local-only (not on
# GitHub), so the lab is bootstrapped from a source zip stored in Drive.
# Build the zip locally with scripts/make_colab_zip.sh and upload it to
# MyDrive/DhakaFlow-Nexus-Data/Lab01_SUMO/lab01_source.zip
DRIVE_OUT = Path('/content/drive/MyDrive/DhakaFlow-Nexus-Data/Lab01_SUMO')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
LAB_ROOT = Path('/content/DhakaFlow-Nexus/paper_labs/01_sumo_foundation')
if not LAB_ROOT.exists():
    src_zip = DRIVE_OUT / 'lab01_source.zip'
    assert src_zip.exists(), (
        f'Lab source not found. Upload {src_zip.name} to {src_zip.parent} '
        '(build it locally with scripts/make_colab_zip.sh)')
    LAB_ROOT.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src_zip) as z:
        z.extractall(LAB_ROOT.parent)
assert LAB_ROOT.exists(), f'Lab root not found: {LAB_ROOT}'
print('LAB_ROOT=', LAB_ROOT)
print('DRIVE_OUT=', DRIVE_OUT)

## Install the complete pinned Colab runtime
This installs `eclipse-sumo==1.27.1` plus Lab 01 Python dependencies. Nothing needs to be installed on the Mac.


In [ ]:
!bash {LAB_ROOT}/scripts/install_colab.sh


In [ ]:
import sys, os
sys.path.insert(0, str(LAB_ROOT / 'src'))
from dhakaflow_lab01.colab_env import configure_sumo_env
env = configure_sumo_env()
env


In [ ]:
!python {LAB_ROOT}/scripts/verify_environment.py
!sumo --version | head -n 3
!netconvert --version | head -n 3


## Static/unit tests and deterministic network build


In [ ]:
!cd {LAB_ROOT} && pytest -q
!bash {LAB_ROOT}/scripts/build_unit_network.sh


## Smoke matrix — same demand/scenario, controller comparison


In [ ]:
!python {LAB_ROOT}/scripts/run_matrix.py --config {LAB_ROOT}/configs/smoke.yaml --output-root {DRIVE_OUT}/unit_smoke


## Full multi-seed matrix


In [ ]:
!python {LAB_ROOT}/scripts/run_matrix.py --config {LAB_ROOT}/configs/full.yaml --output-root {DRIVE_OUT}/unit_full


## Fetch pinned official sources and run Bologna/Acosta unmodified first
Upstream repositories live temporarily under `/content/dhakaflow_upstreams`; heavy evidence goes to Drive.


In [ ]:
os.environ['DHAKAFLOW_UPSTREAM_ROOT'] = '/content/dhakaflow_upstreams/lab01'
!bash {LAB_ROOT}/scripts/fetch_upstreams.sh
!bash {LAB_ROOT}/scripts/run_bologna_unmodified.sh {DRIVE_OUT}/bologna_unmodified


## Final evidence gate
Claude must inspect logs/metrics, independently recompute checks, copy only lightweight evidence to Git, document every modification, and complete `REPRODUCTION_REPORT.md`. Real execution is required before PASS.
